<a href="https://colab.research.google.com/github/davidlealo/practicos_sisrec_2026/blob/main/practico08/07_deepfm_respuestaDLO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctico de DeepFM

# 1. Introducción

En este práctico implementaremos el modelo **DeepFM**, propuesto en el paper [DeepFM: A Factorization-Machine based Neural Network for CTR Prediction](https://arxiv.org/abs/1703.04247).   Para ello utilizaremos la librería **DeepCTR-torch**, que incluye distintos modelos de recomendación basados en redes neuronales.  

El dataset a utilizar será **MovieLens-100k**.  

En cuanto a la evaluación, algunas métricas (como *Precision@K* y *Recall@K*) serán implementadas manualmente, mientras que otras métricas se calcularán con **DeepCTR-torch**.

# 2. Instalación de librerías

In [1]:
!pip install deepctr-torch torch pandas numpy scikit-learn recommenders

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 1.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 625.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.3/355.3 kB 771.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 583.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.6/29.6 MB 425.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 435.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.9/386.9 kB 616.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 477.0 k

# 3. Recomendación de películas sin el género

Importar librerías

In [41]:
from recommenders.datasets import movielens
from deepctr_torch.inputs import SparseFeat, VarLenSparseFeat, get_feature_names
from deepctr_torch.models import DeepFM
from sklearn.model_selection import train_test_split

import numpy as np
import pandas as pd

Descargar dataset

In [42]:
df = movielens.load_pandas_df(
    size="100k",
    genres_col="genre",
    header=["userID", "itemID", "rating"]
)

print(df.shape)
df.sample(5, random_state=42)

100%|██████████| 4.81k/4.81k [00:00<00:00, 22.8kKB/s]


(100000, 4)


,userID,itemID,rating,genre
75721,877,381,4.0,Comedy|Romance
80184,815,602,3.0,Musical|Romance
19864,94,431,4.0,Action|Adventure
76699,416,875,2.0,Drama|Romance
92991,500,182,2.0,Crime|Drama


En **DeepCTR-torch**, el modelo DeepFM está diseñado originalmente para predecir *Click-Through Rate (CTR)*, por lo que la variable objetivo debe ser **binaria**:  
- `0` si el usuario no realizó la acción (no clickeó),  
- `1` si el usuario sí la realizó (clickeó).  

En el caso del dataset de MovieLens, utilizamos las calificaciones como aproximación al interés del usuario. Como el promedio de ratings es cercano a **3.5**, definimos el umbral igual a **4**:  
- ratings `≥ 4` se consideran interacciones positivas (`1`),  
- ratings `< 4` se consideran negativas (`0`).


In [43]:
df["rating"].mean()

np.float64(3.52986)

In [44]:
df = df.copy()
df["label"] = (df["rating"] >= 4).astype(int)

Seleccionamos el primer género como el género principal de la película y el que nos ayudará a la recomendación

In [45]:
def primary_genre(g):
    if isinstance(g, str):
        return g.split("|")[0] if "|" in g else g
    return "Unknown"
df["primary_genre"] = df["genre"].apply(primary_genre).fillna("Unknown")

Se crean diccionarios que convierten cada identificador de usuario, película y género a un índice entero.

Esto es necesario porque los modelos como DeepFM trabajan con índices enteros que luego son pasados a embeddings.

In [46]:
uid2idx = {u:i for i, u in enumerate(sorted(df["userID"].unique()))}
iid2idx = {m:i for i, m in enumerate(sorted(df["itemID"].unique()))}
gid2idx = {g:i for i, g in enumerate(sorted(df["primary_genre"].unique()))}

Se reemplazan los IDs originales por sus índices correspondientes.

In [47]:
df["user_id"] = df["userID"].map(uid2idx).astype(int)
df["item_id"] = df["itemID"].map(iid2idx).astype(int)
df["genre_id"] = df["primary_genre"].map(gid2idx).astype(int)

Printeamos el número total de usuarios, películas y géneros únicos.

In [48]:
n_users = df["user_id"].nunique()
n_items = df["item_id"].nunique()
n_genres = df["genre_id"].nunique()
print(n_users, n_items, n_genres)

943 1682 19


Dividimos el dataset en tres subconjuntos:

- **Entrenamiento (80%)**: se utiliza para entrenar el modelo
- **Validación (10%)**: permite evaluar el desempeño durante el entrenamiento y ajustar hiperparámetros.  
- **Test (10%)**: se reserva para la evaluación final de métricas

In [49]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
val_df, test_df  = train_test_split(val_df, test_size=0.5, random_state=42, stratify=val_df["label"])

En esta celda definimos las **features categóricas** que usará el modelo:

- **`user_id`**: representa a cada usuario del dataset.  
- **`item_id`**: representa a cada ítem o película.  
- **`genre_id`**: representa el género principal de cada película.  

Cada una se declara como un `SparseFeat`, lo que significa que serán tratadas como **variables categóricas** y proyectadas a un **embedding de x dimensiones**.  
De esta manera, el modelo aprende representaciones densas (vectores) tanto para usuarios como para ítems y géneros, capturando relaciones latentes entre ellos.


In [50]:
user_feat = SparseFeat("user_id",  vocabulary_size=n_users,  embedding_dim=16)
item_feat = SparseFeat("item_id",  vocabulary_size=n_items,  embedding_dim=16)

In [51]:
user_feat

SparseFeat(name='user_id', vocabulary_size=943, embedding_dim=16, use_hash=False, dtype='int32', embedding_name='user_id', group_name='default_group')

In [52]:
item_feat

SparseFeat(name='item_id', vocabulary_size=1682, embedding_dim=16, use_hash=False, dtype='int32', embedding_name='item_id', group_name='default_group')

En esta celda se definen las columnas de features para cada parte del modelo:

- **`linear_feature_columns`**: features que alimentan la parte **lineal** (Factorization Machine).  
- **`dnn_feature_columns`**: features que alimentan la parte **no lineal** (red neuronal profunda, DNN).  

En este caso, ambas listas contienen las **mismas variables** (`user_id`, `item_id`, `genre_id`).  
Esto significa que tanto la parte lineal como la no lineal del modelo trabajarán sobre la **misma información de entrada**, pero cada componente la procesará de manera distinta.  

Finalmente, con **`get_feature_names`** se obtiene la lista consolidada de todas las features, necesaria para preparar los datos de entrada al modelo.

In [53]:
linear_feature_columns = [user_feat, item_feat]
dnn_feature_columns = [user_feat, item_feat]
feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)

Se define la función **`build_X`**, que recibe un DataFrame y devuelve un diccionario con las columnas especificadas en `feature_names`.  
  - Cada clave del diccionario corresponde al nombre de una feature (`user_id`, `item_id`, `genre_id`).  
  - Cada valor es un array con los índices de esa feature.  
  - Este formato es el requerido por **DeepCTR-torch** para entrenar el modelo.


In [54]:
def build_X(df_):
    return {name: df_[name].values for name in feature_names if name in df_.columns}

Luego se construyen los conjuntos de datos:  
- **`X_train`, `y_train`**: features y etiquetas del conjunto de entrenamiento.  
- **`X_val`, `y_val`**: features y etiquetas del conjunto de validación.  
- **`X_test`, `y_test`**: features y etiquetas del conjunto de testeo.

De esta forma, los datos quedan listos para ser entregados al modelo DeepFM.

In [55]:
X_train, y_train = build_X(train_df), train_df["label"].values
X_val, y_val = build_X(val_df), val_df["label"].values
X_test, y_test = build_X(test_df), test_df["label"].values

In [56]:
X_train

{'user_id': array([839,  89, 871, ..., 144, 127,  81]),
 'item_id': array([494, 219, 257, ..., 471, 173, 210])}

In [57]:
y_train

array([0, 1, 1, ..., 0, 0, 1])

In [58]:
X_val

{'user_id': array([384, 346, 832, ..., 302, 516, 304]),
 'item_id': array([ 98, 870, 825, ..., 180, 747, 195])}

In [59]:
y_val

array([0, 1, 0, ..., 1, 1, 1])

In [60]:
X_test

{'user_id': array([906, 721, 304, ..., 614, 492, 528]),
 'item_id': array([143, 123,  65, ..., 643, 123, 983])}

In [61]:
y_test

array([1, 1, 0, ..., 1, 0, 1])

En esta celda se instancia el modelo **DeepFM** usando la implementación de *DeepCTR-torch*.  
Los parámetros principales son:

- **`linear_feature_columns` y `dnn_feature_columns`**: definen las features de entrada para la parte lineal (FM) y para la red neuronal (DNN).  
- **`task='binary'`**: indica que el modelo resolverá un problema de clasificación binaria (en este caso, interacción relevante o no relevante).  
- **`dnn_hidden_units=(128, 64)`**: la parte de red neuronal profunda tendrá dos capas ocultas, la primera con 128 neuronas y la segunda con 64.  
- **`dnn_dropout=0.2`**: aplica *dropout* del 20% en las capas de la DNN para reducir sobreajuste.  
- **`device="cpu"`**: especifica que el entrenamiento se realizará en CPU (podría configurarse `"cuda"` para usar GPU).

De esta forma, el modelo DeepFM combina la parte **lineal (Factorization Machines)** y la parte **no lineal (red neuronal profunda)** en un solo modelo de recomendación.


In [62]:
model = DeepFM(
    linear_feature_columns, dnn_feature_columns,
    task='binary', dnn_hidden_units=(128,64), dnn_dropout=0.2, device="cpu"
)

En esta celda se compila el modelo DeepFM, definiendo cómo será entrenado:

- **`optimizer="adam"`**: se utiliza el [optimizador Adam](https://arxiv.org/abs/1412.6980).
- **`loss="binary_crossentropy"`**: la función de pérdida adecuada para problemas de clasificación binaria.
- **`metrics=["auc"]`**: además de la pérdida, se calculará el **AUC**. Esta métrica es común en tareas de predicción de CTR, ya que mide la capacidad del modelo para rankear correctamente ejemplos positivos frente a negativos.

En **DeepCTR-torch**, las métricas que pueden configurarse en `metrics=[...]` son: "auc", "mse" y ,"accuracy". Más información en el método _get_metrics de la [documentación](https://deepctr-torch.readthedocs.io/en/latest/_modules/deepctr_torch/models/basemodel.html#BaseModel.compile).

De esta forma, se establecen tanto la función de optimización como las métricas de evaluación que se usarán durante el entrenamiento del modelo.


In [63]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["auc"])

En esta celda se entrena el modelo DeepFM con el método **`fit`**, indicando los parámetros de entrenamiento:

- **`X_train, y_train`**: datos de entrada y etiquetas del conjunto de entrenamiento.  
- **`batch_size=2048`**: cantidad de muestras que se procesan en cada iteración antes de actualizar los parámetros.  
- **`epochs=10`**: número de veces que el modelo recorre por completo el conjunto de entrenamiento.  
- **`verbose=2`**: controla el nivel de detalle mostrado durante el entrenamiento (2 muestra una línea por época).  
- **`validation_data=(X_val, y_val)`**: conjunto de validación que se evalúa al final de cada época para monitorear el desempeño del modelo y detectar sobreajuste.

In [64]:
model.fit(X_train, y_train, batch_size=2048, epochs=10, verbose=2, validation_data=(X_val, y_val))

cpu
Train on 80000 samples, validate on 10000 samples, 40 steps per epoch
Epoch 1/10
1s - loss:  0.6838 - auc:  0.6202 - val_auc:  0.7557
Epoch 2/10
2s - loss:  0.6449 - auc:  0.7698 - val_auc:  0.7598
Epoch 3/10
1s - loss:  0.5861 - auc:  0.7818 - val_auc:  0.7665
Epoch 4/10
1s - loss:  0.5634 - auc:  0.7921 - val_auc:  0.7695
Epoch 5/10
1s - loss:  0.5537 - auc:  0.7949 - val_auc:  0.7707
Epoch 6/10
1s - loss:  0.5486 - auc:  0.7957 - val_auc:  0.7712
Epoch 7/10
1s - loss:  0.5456 - auc:  0.7975 - val_auc:  0.7723
Epoch 8/10
1s - loss:  0.5439 - auc:  0.7986 - val_auc:  0.7724
Epoch 9/10
1s - loss:  0.5425 - auc:  0.7995 - val_auc:  0.7730
Epoch 10/10
1s - loss:  0.5412 - auc:  0.7986 - val_auc:  0.7733


Definimos las funciones para evaluación Precision@K y Recall@K

In [65]:
def precision_recall_at_k(df_user, k):
    df_user = df_user.sort_values('score', ascending=False)
    topk = df_user.head(k)
    hits_k = int(topk['label'].sum())
    total_rel = int(df_user['label'].sum())

    prec = hits_k / k
    rec = hits_k / total_rel if total_rel > 0 else np.nan

    return prec, rec, total_rel

def macro_precision_recall_at_k(df, k):
    precs, recs = [], []
    for uid, df_u in df.groupby('user_id'):
        p, r, tot_rel = precision_recall_at_k(df_u, k)
        precs.append(p)
        if not np.isnan(r):
            recs.append(r)

    return float(np.mean(precs)), (float(np.mean(recs)) if len(recs) > 0 else np.nan)

Se generan las predicciones del modelo sobre el conjunto de **test**:

- **`model.predict(X_test, batch_size=4096)`**: obtiene los puntajes de predicción para cada par usuario–ítem del conjunto de test.  
- **`.reshape(-1)`**: ajusta el formato de salida a un vector unidimensional.  
- **`test_eval = test_df[['user_id','item_id','label']].copy()`**: se crea un DataFrame de evaluación que contiene los identificadores de usuario, ítem y la etiqueta real (`label`).  
- **`test_eval['score'] = y_scores`**: se agrega una nueva columna con los puntajes predichos por el modelo.

In [66]:
y_scores = model.predict(X_test, batch_size=4096).reshape(-1)
test_eval = test_df[['user_id','item_id','label']].copy()
test_eval['score'] = y_scores

Calculamos las métricas para K = 5, 10 y 20. Guardamos sus resultados en un dataframe para mostrarlo más fácilmente

In [67]:
K_list = [5, 10, 20]
rows = []
for K in K_list:
    p, r = macro_precision_recall_at_k(test_eval, K)
    rows.append({
        'K': K,
        'Precision@K (macro)': p,
        'Recall@K (macro)': r,
    })

metrics_df = pd.DataFrame(rows)
metrics_df

,K,Precision@K (macro),Recall@K (macro)
0,5,0.558887,0.702980
1,10,0.418522,0.871136
2,20,0.270610,0.971696


# 4. Recomendación de películas agregando el género

Se define la función **`split_genres`**, que procesa la columna de géneros de cada película:

- Si el valor es un string válido (`"Comedy|Romance|Drama"`), lo divide en una **lista de géneros individuales**, separando por el carácter `"|"`.  
- Si el valor es nulo o no es un string, asigna el género `"Unknown"`.  

De esta forma, cada película queda representada no por un único género principal, sino por una **lista de todos sus géneros asociados**.

In [68]:
def split_genres(g):
    if isinstance(g, str) and g.strip():
        return [x.strip() for x in g.split("|")]
    return ["Unknown"]

In [69]:
df["genres_list"] = df["genre"].apply(split_genres)

Se construye el **vocabulario de géneros** y se define el esquema de codificación numérica:

- **`all_genres`**: contiene la lista ordenada de todos los géneros únicos presentes en el dataset.  
- **`genre2id`**: asigna a cada género un identificador numérico entero, comenzando en `1`.  
  - El `+1` se utiliza porque el índice `0` se reserva como **padding**.  
- **`PAD_ID = 0`**: valor especial que representa el padding y se usa para completar secuencias de géneros a una longitud fija.  
- **`VOCAB_SIZE_GENRE = len(all_genres) + 1`**: tamaño total del vocabulario, incluyendo la posición adicional para el padding.

De esta forma, cada género puede mapearse a un número entero, y las secuencias de géneros pueden representarse con longitudes uniformes.

In [70]:
all_genres = sorted({g for lst in df["genres_list"] for g in lst})
genre2id = {g: i+1 for i, g in enumerate(all_genres)}
PAD_ID = 0
VOCAB_SIZE_GENRE = len(all_genres) + 1

In [71]:
df["genres_ids"] = df["genres_list"].apply(lambda lst: [genre2id[g] for g in lst])

Se define la función **`pad_seq`**, utilizada para uniformar la longitud de las secuencias de géneros:

- **`MAXLEN = 5`**: se fija la longitud máxima de las secuencias.  
- La función **`pad_seq(ids, maxlen, pad_value)`** recibe una lista de identificadores de géneros:  
  - Si la lista es más larga que `MAXLEN`, se **trunca** a la longitud máxima.  
  - Si es más corta, se **rellena con el valor de padding (`PAD_ID = 0`)** hasta alcanzar `MAXLEN`.  

De esta forma, todas las películas quedan representadas con una secuencia de géneros de tamaño fijo (`MAXLEN`), lo que permite procesarlas en `deepctr-torch`.

In [72]:
MAXLEN = 3

In [73]:
def pad_seq(ids, maxlen=MAXLEN, pad_value=PAD_ID):
    ids = ids[:maxlen]
    return ids + [pad_value] * (maxlen - len(ids))

Se crean las columnas con el padding, "genres_seq_len" almacena la longitud real de cada secuencia

In [74]:
df["genres_seq"] = df["genres_ids"].apply(lambda ids: pad_seq(ids, MAXLEN, PAD_ID))
df["genres_seq_len"] = df["genres_ids"].apply(lambda ids: min(len(ids), MAXLEN))

En esta celda se define el feature secuencial de géneros y se incorporan todas las features al modelo:

- Se utiliza `VarLenSparseFeat` para representar la lista de géneros de cada película como una secuencia.  
- **`SparseFeat("genres_seq", vocabulary_size=VOCAB_SIZE_GENRE, embedding_dim=16)`**: cada género se convierte en un embedding de 16 dimensiones.  
- **`maxlen=MAXLEN`**: fija la longitud máxima de la secuencia (con padding si es necesario).  
- **`combiner="mean"`**: combina los embeddings de todos los géneros de una película calculando su promedio.  
- **`length_name="genres_seq_len"`**: indica la longitud real de la secuencia para que el modelo ignore el padding.


In [75]:
genre_seq_feat = VarLenSparseFeat(
    SparseFeat("genres_seq", vocabulary_size=VOCAB_SIZE_GENRE, embedding_dim=16),
    maxlen=MAXLEN,
    combiner="mean",
    length_name="genres_seq_len"
)

linear_feature_columns = [user_feat, item_feat, genre_seq_feat]
dnn_feature_columns    = [user_feat, item_feat, genre_seq_feat]

feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)

Se define la función **`build_X`**, que prepara los datos de entrada en el formato requerido por *DeepCTR-torch*.

El resultado es un diccionario `X` que contiene todos los datos en el formato esperado por el modelo DeepFM, listo para ser usado en el entrenamiento y la evaluación.


In [76]:
def build_X(df_):
    X = {}

    for name in feature_names:
        if name in df_.columns:
            X[name] = df_[name].values

    X["genres_seq"]     = np.stack(df_["genres_seq"].values)
    X["genres_seq_len"] = df_["genres_seq_len"].values

    return X

Se sigue el mismo proceso que al entrenar sin género

In [77]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
val_df, test_df  = train_test_split(val_df, test_size=0.5, random_state=42, stratify=val_df["label"])

In [78]:
train_df

,userID,itemID,rating,genre,label,primary_genre,user_id,item_id,genre_id,genres_list,genres_ids,genres_seq,genres_seq_len
63456,840,495,3.0,Adventure|Comedy,0,Adventure,839,494,1,"[Adventure, Comedy]","[2, 5]","[2, 5, 0]",2
69727,90,220,4.0,Comedy|Romance,1,Comedy,89,219,4,"[Comedy, Romance]","[5, 14]","[5, 14, 0]",2
60651,872,258,4.0,Drama|Sci-Fi,1,Drama,871,257,7,"[Drama, Sci-Fi]","[8, 15]","[8, 15, 0]",2
40113,450,26,5.0,Comedy,1,Comedy,449,25,4,[Comedy],[5],"[5, 0, 0]",1
20913,294,122,3.0,Comedy,0,Comedy,293,121,4,[Comedy],[5],"[5, 0, 0]",1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
91921,13,870,3.0,Romance,0,Romance,12,869,13,[Romance],[14],"[14, 0, 0]",1
17159,128,505,4.0,Mystery|Thriller,1,Mystery,127,504,12,"[Mystery, Thriller]","[13, 16]","[13, 16, 0]",2
1762,145,472,3.0,Action|Adventure|Fantasy,0,Action,144,471,0,"[Action, Adventure, Fantasy]","[1, 2, 9]","[1, 2, 9]",3
55713,128,174,3.0,Action|Adventure,0,Action,127,173,0,"[Action, Adventure]","[1, 2]","[1, 2, 0]",2


In [79]:
val_df

,userID,itemID,rating,genre,label,primary_genre,user_id,item_id,genre_id,genres_list,genres_ids,genres_seq,genres_seq_len
11079,385,99,2.0,Animation|Children's|Musical,0,Animation,384,98,2,"[Animation, Children's, Musical]","[3, 4, 12]","[3, 4, 12]",3
14856,347,871,4.0,Comedy,1,Comedy,346,870,4,[Comedy],[5],"[5, 0, 0]",1
71138,833,826,2.0,Adventure,0,Adventure,832,825,1,[Adventure],[2],"[2, 0, 0]",1
26905,480,319,3.0,Comedy|Musical|Romance,0,Comedy,479,318,4,"[Comedy, Musical, Romance]","[5, 12, 14]","[5, 12, 14]",3
20651,18,6,5.0,Drama,1,Drama,17,5,7,[Drama],[8],"[8, 0, 0]",1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
36426,379,271,3.0,Action|Adventure|Sci-Fi|War,0,Action,378,270,0,"[Action, Adventure, Sci-Fi, War]","[1, 2, 15, 17]","[1, 2, 15]",3
7260,320,678,3.0,Drama|Thriller,0,Drama,319,677,7,"[Drama, Thriller]","[8, 16]","[8, 16, 0]",2
5815,303,181,5.0,Action|Adventure|Romance|Sci-Fi|War,1,Action,302,180,0,"[Action, Adventure, Romance, Sci-Fi, War]","[1, 2, 14, 15, 17]","[1, 2, 14]",3
83468,517,748,4.0,Action|Romance|Thriller,1,Action,516,747,0,"[Action, Romance, Thriller]","[1, 14, 16]","[1, 14, 16]",3


In [80]:
test_df

,userID,itemID,rating,genre,label,primary_genre,user_id,item_id,genre_id,genres_list,genres_ids,genres_seq,genres_seq_len
82665,907,144,5.0,Action|Thriller,1,Action,906,143,0,"[Action, Thriller]","[1, 16]","[1, 16, 0]",2
46760,722,124,4.0,Drama|Mystery,1,Drama,721,123,7,"[Drama, Mystery]","[8, 13]","[8, 13, 0]",2
81723,305,66,3.0,Comedy|Romance,0,Comedy,304,65,4,"[Comedy, Romance]","[5, 14]","[5, 14, 0]",2
80909,901,259,2.0,Children's|Comedy,0,Children's,900,258,3,"[Children's, Comedy]","[4, 5]","[4, 5, 0]",2
27021,91,603,5.0,Mystery|Thriller,1,Mystery,90,602,12,"[Mystery, Thriller]","[13, 16]","[13, 16, 0]",2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13324,345,471,3.0,Drama|War,0,Drama,344,470,7,"[Drama, War]","[8, 17]","[8, 17, 0]",2
69892,11,729,4.0,Drama,1,Drama,10,728,7,[Drama],[8],"[8, 0, 0]",1
36880,615,644,4.0,Documentary,1,Documentary,614,643,6,[Documentary],[7],"[7, 0, 0]",1
58319,493,124,3.0,Drama|Mystery,0,Drama,492,123,7,"[Drama, Mystery]","[8, 13]","[8, 13, 0]",2


In [81]:
X_train, y_train = build_X(train_df), train_df["label"].values
X_val, y_val = build_X(val_df), val_df["label"].values
X_test, y_test = build_X(test_df), test_df["label"].values

In [82]:
X_train

{'user_id': array([839,  89, 871, ..., 144, 127,  81]),
 'item_id': array([494, 219, 257, ..., 471, 173, 210]),
 'genres_seq': array([[ 2,  5,  0],
        [ 5, 14,  0],
        [ 8, 15,  0],
        ...,
        [ 1,  2,  9],
        [ 1,  2,  0],
        [ 5, 17,  0]]),
 'genres_seq_len': array([2, 2, 2, ..., 3, 2, 2])}

In [83]:
model = DeepFM(
    linear_feature_columns, dnn_feature_columns,
    task='binary', dnn_hidden_units=(128,64), dnn_dropout=0.2, device="cpu"
)

In [84]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["auc"])

In [85]:
model.fit(X_train, y_train, batch_size=2048, epochs=10, verbose=2, validation_data=(X_val, y_val))

cpu
Train on 80000 samples, validate on 10000 samples, 40 steps per epoch
Epoch 1/10
1s - loss:  0.6817 - auc:  0.6435 - val_auc:  0.7481
Epoch 2/10
1s - loss:  0.6280 - auc:  0.7683 - val_auc:  0.7617
Epoch 3/10
1s - loss:  0.5772 - auc:  0.7810 - val_auc:  0.7663
Epoch 4/10
1s - loss:  0.5600 - auc:  0.7888 - val_auc:  0.7684
Epoch 5/10
1s - loss:  0.5519 - auc:  0.7943 - val_auc:  0.7700
Epoch 6/10
1s - loss:  0.5478 - auc:  0.7955 - val_auc:  0.7705
Epoch 7/10
2s - loss:  0.5454 - auc:  0.7971 - val_auc:  0.7710
Epoch 8/10
2s - loss:  0.5438 - auc:  0.7962 - val_auc:  0.7716
Epoch 9/10
1s - loss:  0.5426 - auc:  0.7972 - val_auc:  0.7710
Epoch 10/10
1s - loss:  0.5418 - auc:  0.7971 - val_auc:  0.7719


In [86]:
y_scores = model.predict(X_test, batch_size=4096).reshape(-1)
test_eval = test_df[["user_id","item_id","label"]].copy()
test_eval["score"] = y_scores

In [87]:
K_list = [5, 10, 20]
rows = []
for K in K_list:
    p, r = macro_precision_recall_at_k(test_eval, K)
    rows.append({
        'K': K,
        'Precision@K (macro)': p,
        'Recall@K (macro)': r,
    })

metrics_df = pd.DataFrame(rows)
metrics_df

,K,Precision@K (macro),Recall@K (macro)
0,5,0.559529,0.704518
1,10,0.417559,0.870710
2,20,0.270557,0.970563


In [88]:
df

,userID,itemID,rating,genre,label,primary_genre,user_id,item_id,genre_id,genres_list,genres_ids,genres_seq,genres_seq_len
0,196,242,3.0,Comedy,0,Comedy,195,241,4,[Comedy],[5],"[5, 0, 0]",1
1,186,302,3.0,Crime|Film-Noir|Mystery|Thriller,0,Crime,185,301,5,"[Crime, Film-Noir, Mystery, Thriller]","[6, 10, 13, 16]","[6, 10, 13]",3
2,22,377,1.0,Children's|Comedy,0,Children's,21,376,3,"[Children's, Comedy]","[4, 5]","[4, 5, 0]",2
3,244,51,2.0,Drama|Romance|War|Western,0,Drama,243,50,7,"[Drama, Romance, War, Western]","[8, 14, 17, 18]","[8, 14, 17]",3
4,166,346,1.0,Crime|Drama,0,Crime,165,345,5,"[Crime, Drama]","[6, 8]","[6, 8, 0]",2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,880,476,3.0,Comedy,0,Comedy,879,475,4,[Comedy],[5],"[5, 0, 0]",1
99996,716,204,5.0,Comedy|Sci-Fi,1,Comedy,715,203,4,"[Comedy, Sci-Fi]","[5, 15]","[5, 15, 0]",2
99997,276,1090,1.0,Thriller,0,Thriller,275,1089,15,[Thriller],[16],"[16, 0, 0]",1
99998,13,225,2.0,Children's|Comedy,0,Children's,12,224,3,"[Children's, Comedy]","[4, 5]","[4, 5, 0]",2


In [89]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 13 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   userID          100000 non-null  int64  
 1   itemID          100000 non-null  int64  
 2   rating          100000 non-null  float64
 3   genre           100000 non-null  object 
 4   label           100000 non-null  int64  
 5   primary_genre   100000 non-null  object 
 6   user_id         100000 non-null  int64  
 7   item_id         100000 non-null  int64  
 8   genre_id        100000 non-null  int64  
 9   genres_list     100000 non-null  object 
 10  genres_ids      100000 non-null  object 
 11  genres_seq      100000 non-null  object 
 12  genres_seq_len  100000 non-null  int64  
dtypes: float64(1), int64(7), object(5)
memory usage: 9.9+ MB


In [96]:
# Valores únicos de userID
df['userID'].nunique()

943

In [97]:
df['itemID'].nunique()

1682

In [98]:
df['rating'].nunique()

5

In [99]:
df['genre'].nunique()

216

In [100]:
df['label'].nunique()

2

In [101]:
df['primary_genre'].nunique()

19

In [102]:
df['user_id'].nunique()

943

In [103]:
df['item_id'].nunique()

1682

# 5. Actividad

# Parte 1 (2 puntos)

Explique por qué la incorporación de los géneros produce una mejoría casi nula en las métricas.

**Respuesta**

La inclusión de los géneros no parece ser clave en el cambio de la predicción, estuve analizando los DFs y las variables antes de ingresar en los modelos para el entrenamiento.

Entonces, al parecer el género no cambia mucho la predicción que hacemos del rating y eso, puede tener sentido: por ejemplo, puede existir una película de comedia de 5 estrellas y otra que también es comedia con solamente una, entonces el género o los géneros que tiene el ítem no cambia su calidad para un usuario, por eso creo que en este caso no cambia la predicción de forma significativa, porque al final estamos evaluando el mismo ítem.

# Parte 2 (2 puntos)

Lea el abstract y la introducción del paper de [DeepFM](https://arxiv.org/abs/1703.04247). Con esta información, explique con sus palabras de qué se trata el modelo DeepFM y cuál es su relación con las FM tradicionales

**Respuesta**

El paper propone DeepFM como solución a un problema concreto: los modelos existentes para predecir CTR, que se refiere a la predicción del click-through rate.

Los modelos anteriores tenían un sesgo hacia un solo tipo de interacción entre features, ya sea interacciones simples (orden bajo, como FM tradicional) o complejas (orden alto, como redes neuronales profundas), y los que intentaban combinar ambas como Wide & Deep de Google requerían ingeniería manual de features.

DeepFM resuelve esto combinando una Factorization Machine y una red neuronal profunda en paralelo, donde ambas partes comparten exactamente los mismos embeddings de entrada, permitiendo capturar interacciones de todos los órdenes de forma automática y sin necesidad de diseñar features manualmente.

La relación con las FM tradicionales es directa: DeepFM mantiene la componente FM intacta para capturar interacciones de segundo orden, pero le agrega la DNN encima para capturar patrones más complejos, resultando en un modelo más expresivo que una FM sola pero sin perder sus ventajas.

**Respuesta**



# Parte 3 (4 puntos)

Elija una de las dos versiones de DeepFM entrenadas (con o sin géneros de películas). Modifique los parámetros "batch size" y/o "epochs". Intente al menos 3 combinaciones y explique la diferencia entre el desempeño (en caso de no haber diferencia, explique la causa)

In [106]:
combinaciones = [
     {"batch_size": 256,  "epochs": 10},
    {"batch_size": 1024, "epochs": 10},
    {"batch_size": 4096, "epochs": 10},
]

In [108]:
resultados = []

for params in combinaciones:
    print(f"\n{'='*50}")
    print(f"batch_size={params['batch_size']}, epochs={params['epochs']}")
    print(f"{'='*50}")

    model = DeepFM(
        linear_feature_columns, dnn_feature_columns,
        task='binary', dnn_hidden_units=(128, 64), dnn_dropout=0.2, device="cpu"
    )

    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["auc"])

    model.fit(
        X_train, y_train,
        batch_size=params['batch_size'],
        epochs=params['epochs'],
        verbose=2,
        validation_data=(X_val, y_val)
    )

    y_scores = model.predict(X_test, batch_size=4096).reshape(-1)
    test_eval = test_df[["user_id", "item_id", "label"]].copy()
    test_eval["score"] = y_scores

    for K in [5, 10, 20]:
        p, r = macro_precision_recall_at_k(test_eval, K)
        resultados.append({
            "batch_size": params['batch_size'],
            "epochs":     params['epochs'],
            "K":          K,
            "Precision@K": round(p, 4),
            "Recall@K":    round(r, 4),
        })


batch_size=256, epochs=10
cpu
Train on 80000 samples, validate on 10000 samples, 313 steps per epoch
Epoch 1/10
3s - loss:  0.6148 - auc:  0.7243 - val_auc:  0.7665
Epoch 2/10
3s - loss:  0.5604 - auc:  0.7806 - val_auc:  0.7713
Epoch 3/10
5s - loss:  0.5526 - auc:  0.7867 - val_auc:  0.7722
Epoch 4/10
3s - loss:  0.5488 - auc:  0.7892 - val_auc:  0.7741
Epoch 5/10
3s - loss:  0.5440 - auc:  0.7932 - val_auc:  0.7760
Epoch 6/10
4s - loss:  0.5370 - auc:  0.7986 - val_auc:  0.7783
Epoch 7/10
3s - loss:  0.5310 - auc:  0.8031 - val_auc:  0.7783
Epoch 8/10
3s - loss:  0.5267 - auc:  0.8062 - val_auc:  0.7786
Epoch 9/10
3s - loss:  0.5230 - auc:  0.8094 - val_auc:  0.7763
Epoch 10/10
3s - loss:  0.5197 - auc:  0.8120 - val_auc:  0.7756

batch_size=1024, epochs=10
cpu
Train on 80000 samples, validate on 10000 samples, 79 steps per epoch
Epoch 1/10
2s - loss:  0.6635 - auc:  0.6816 - val_auc:  0.7582
Epoch 2/10
1s - loss:  0.5800 - auc:  0.7761 - val_auc:  0.7680
Epoch 3/10
2s - loss:  0.55

In [109]:
resultados_df = pd.DataFrame(resultados)
resultados_df

,batch_size,epochs,K,Precision@K,Recall@K
0,256,10,5,0.5574,0.7033
1,256,10,10,0.4182,0.8709
2,256,10,20,0.2703,0.9705
3,1024,10,5,0.5587,0.7022
4,1024,10,10,0.4184,0.8706
5,1024,10,20,0.2708,0.9709
6,4096,10,5,0.5606,0.7026
7,4096,10,10,0.4179,0.8706
8,4096,10,20,0.2707,0.9709


Las tres combinaciones producen métricas casi idénticas, este es un dataset simple, no muy extenso y que estamos usando el mismo optimizador. Al crecer el batch size mejora un poquito mejor en el precision, pero nada que muestre claramente una mejora.

In [110]:
combinaciones = [
    {"batch_size": 2048, "epochs": 5},
    {"batch_size": 2048, "epochs": 15},
    {"batch_size": 2048, "epochs": 30},
]

resultados2 = []

for params in combinaciones:
    print(f"\n{'='*50}")
    print(f"batch_size={params['batch_size']}, epochs={params['epochs']}")
    print(f"{'='*50}")

    model = DeepFM(
        linear_feature_columns, dnn_feature_columns,
        task='binary', dnn_hidden_units=(128, 64), dnn_dropout=0.2, device="cpu"
    )

    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["auc"])

    model.fit(
        X_train, y_train,
        batch_size=params['batch_size'],
        epochs=params['epochs'],
        verbose=2,
        validation_data=(X_val, y_val)
    )

    y_scores = model.predict(X_test, batch_size=4096).reshape(-1)
    test_eval = test_df[["user_id", "item_id", "label"]].copy()
    test_eval["score"] = y_scores

    for K in [5, 10, 20]:
        p, r = macro_precision_recall_at_k(test_eval, K)
        resultados2.append({
            "batch_size": params['batch_size'],
            "epochs":     params['epochs'],
            "K":          K,
            "Precision@K": round(p, 4),
            "Recall@K":    round(r, 4),
        })




batch_size=2048, epochs=5
cpu
Train on 80000 samples, validate on 10000 samples, 40 steps per epoch
Epoch 1/5
1s - loss:  0.6817 - auc:  0.6435 - val_auc:  0.7481
Epoch 2/5
1s - loss:  0.6280 - auc:  0.7683 - val_auc:  0.7617
Epoch 3/5
1s - loss:  0.5772 - auc:  0.7810 - val_auc:  0.7663
Epoch 4/5
1s - loss:  0.5600 - auc:  0.7888 - val_auc:  0.7684
Epoch 5/5
2s - loss:  0.5519 - auc:  0.7943 - val_auc:  0.7700

batch_size=2048, epochs=15
cpu
Train on 80000 samples, validate on 10000 samples, 40 steps per epoch
Epoch 1/15
1s - loss:  0.6817 - auc:  0.6435 - val_auc:  0.7481
Epoch 2/15
1s - loss:  0.6280 - auc:  0.7683 - val_auc:  0.7617
Epoch 3/15
1s - loss:  0.5772 - auc:  0.7810 - val_auc:  0.7663
Epoch 4/15
1s - loss:  0.5600 - auc:  0.7888 - val_auc:  0.7684
Epoch 5/15
1s - loss:  0.5519 - auc:  0.7943 - val_auc:  0.7700
Epoch 6/15
1s - loss:  0.5478 - auc:  0.7955 - val_auc:  0.7705
Epoch 7/15
2s - loss:  0.5454 - auc:  0.7971 - val_auc:  0.7710
Epoch 8/15
1s - loss:  0.5438 - au

In [111]:
resultados2_df = pd.DataFrame(resultados2)
resultados2_df

,batch_size,epochs,K,Precision@K,Recall@K
0,2048,5,5,0.5604,0.7039
1,2048,5,10,0.4184,0.8712
2,2048,5,20,0.2706,0.9706
3,2048,15,5,0.5591,0.7037
4,2048,15,10,0.4188,0.8719
5,2048,15,20,0.2706,0.9705
6,2048,30,5,0.5580,0.7029
7,2048,30,10,0.4188,0.8718
8,2048,30,20,0.2707,0.9704


Este nuevo experimento cuando cambié tanto el batch size y las epochs, las métricas son prácticamente idénticas, sin ver nuevamente cambios significativos. Incluso hasta menos epoch dejan un poco mejor resultados en precision, lo que me parece interesante, porque efectivamente al subir el k mejora el recall, pero eso también es porque tiene acceso a más datos y es un comportamiento de la relación precision-recall que ya conocemos bien y hemos visto un montón en clases.


# Parte 4 (2 puntos)

Con la mejor combinación de parámetros obtenida en el ítem anterior, modifique la compilación del modelo para incluir todas las métricas disponibles*.  

Para ello, cambie el parámetro `metrics` en el método `model.compile`

In [112]:
model = DeepFM(
    linear_feature_columns, dnn_feature_columns,
    task='binary', dnn_hidden_units=(128, 64), dnn_dropout=0.2, device="cpu"
)

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["auc", "mse", "accuracy"])

model.fit(
    X_train, y_train,
    batch_size=1024,
    epochs=5,
    verbose=2,
    validation_data=(X_val, y_val)
)

cpu
Train on 80000 samples, validate on 10000 samples, 79 steps per epoch
Epoch 1/5
2s - loss:  0.6635 - auc:  0.6816 - mse:  0.2354 - accuracy:  0.5839 - val_auc:  0.7582 - val_mse:  0.2098 - val_accuracy:  0.6973
Epoch 2/5
1s - loss:  0.5800 - auc:  0.7761 - mse:  0.1975 - accuracy:  0.7110 - val_auc:  0.7680 - val_mse:  0.1967 - val_accuracy:  0.7036
Epoch 3/5
1s - loss:  0.5569 - auc:  0.7877 - mse:  0.1876 - accuracy:  0.7216 - val_auc:  0.7698 - val_mse:  0.1938 - val_accuracy:  0.7059
Epoch 4/5
2s - loss:  0.5499 - auc:  0.7912 - mse:  0.1849 - accuracy:  0.7263 - val_auc:  0.7711 - val_mse:  0.1930 - val_accuracy:  0.7079
Epoch 5/5
2s - loss:  0.5469 - auc:  0.7939 - mse:  0.1833 - accuracy:  0.7281 - val_auc:  0.7725 - val_mse:  0.1925 - val_accuracy:  0.7102


Se incluyeron las tres métricas disponibles: auc, mse y accuracy, usando batch_size de 1024 y el número 5 de las epochs.

AUC fue la métrica que más mejoró a lo largo del entrenamiento desde 0.758 a 0.772 en validación, y es la más relevante para este problema porque mide la capacidad del modelo de rankear correctamente los ítems.

MSE bajó desde 0.209 a 0.192, lo que indica que las probabilidades predichas se fueron ajustando mejor y por lo tanto encontramos una mejora también.

Accuracy subió de 0.697 a 0.710, reflejando que el modelo clasifica correctamente el 71% de los pares usuario-ítem.

Las tres métricas muestran una brecha pequeña pero consistente entre entrenamiento y validación. Igual me gustaría tener más tiempo o más datos, porque las mejoras existen, pero no son tan grandes como para obvio tener mejores descubrimientos.